# Phase 4F — controlled BIFOLD S2 head-only adaptation
One deterministic TRAIN/validation run. Test remains sealed.

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO_DIR = Path('/kaggle/working/SIH-26167-SATQuery')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', os.environ['SATQUERY_REPO_URL'], str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[multisensor]', 'appdirs>=1.4.4,<2', 'lightning>=2,<3', 'lmdb==1.6.2', 'timm==0.9.16'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--ignore-requires-python', 'configilm==0.7.0'], check=True)
REBEN_DIR = Path('/kaggle/working/reben-training-scripts')
if not REBEN_DIR.exists():
    subprocess.run(['git', 'clone', 'https://git.tu-berlin.de/rsim/reben-training-scripts.git', str(REBEN_DIR)], check=True)
subprocess.run(['git', 'checkout', '90f7a58a2757bb407df64dd01bfc62b79df2bdd5'], cwd=REBEN_DIR, check=True)
sys.path.insert(0, str(REBEN_DIR))


In [ ]:
import hashlib, json, shutil, tarfile, time
from pathlib import PurePosixPath
import zstandard
from ml.evaluation.phase4e_bifold_baseline import assert_phase4d_ready
from ml.training.phase4f_s2_head import validate_phase4e_baseline_linkage
phase4f_started = time.perf_counter()
EXPERIMENT_DIR = REPO_DIR / 'experiments/phase4_bigearthnet_multisensor'
READINESS = EXPERIMENT_DIR / 'phase4e_readiness.json'
assert_phase4d_ready(READINESS, EXPERIMENT_DIR / 'results/representative_raster_audit.json', EXPERIMENT_DIR / 'bifold_contract.json', EXPERIMENT_DIR / 'split_manifest.json')
readiness = json.loads(READINESS.read_text())
validate_phase4e_baseline_linkage(json.loads((EXPERIMENT_DIR / 'phase4e_closeout.json').read_text()))
matches = list(Path('/kaggle/input').rglob('phase4_s2_selected.tar.zst'))
if len(matches) != 1:
    raise RuntimeError(f'Expected exactly one attached S2 package, found {len(matches)}')
package = matches[0]
digest = hashlib.sha256()
with package.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b''):
        digest.update(chunk)
if digest.hexdigest() != readiness['modalities']['s2']['package_sha256']:
    raise RuntimeError('Attached S2 package SHA-256 mismatch')
DATASET_ROOT = Path('/kaggle/working/phase4f-data')
with package.open('rb') as source, zstandard.ZstdDecompressor().stream_reader(source, read_across_frames=True) as stream, tarfile.open(fileobj=stream, mode='r|') as archive:
    for member in archive:
        path = PurePosixPath(member.name)
        if not member.isfile() or path.is_absolute() or '..' in path.parts:
            raise RuntimeError(f'Unsafe packaged member: {member.name}')
        if len(path.parts) < 2 or path.parts[0] != 's2' or path.parts[1] not in {'train', 'validation'}:
            continue
        target = DATASET_ROOT.joinpath(*path.parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        extracted = archive.extractfile(member)
        if extracted is None:
            raise RuntimeError(f'Cannot extract packaged member: {member.name}')
        with target.open('wb') as output:
            shutil.copyfileobj(extracted, output)


In [ ]:
import platform
import torch
from safetensors.torch import save_file
from torch.utils.data import DataLoader
from configilm.extra.BENv2_utils import NEW_LABELS
from reben_publication.BigEarthNetv2_0_ImageClassifier import BigEarthNetv2_0_ImageClassifier
from ml.evaluation.phase4e_bifold_baseline import BifoldS2Inference, Phase4DGatePaths, Phase4EProvenance, evaluate_validation_batches, iter_validation_batches
from ml.training.phase4f_s2_head import FROZEN_PHASE4E_S2_BASELINE, Phase4FProvenance, Phase4FS2Dataset, Phase4FTrainingConfig, fit_head_only, freeze_s2_classifier_head
if not torch.cuda.is_available():
    raise RuntimeError('Phase 4F requires a Kaggle GPU')
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
if any(torch.cuda.get_device_capability(i)[0] < 7 for i in range(torch.cuda.device_count())):
    raise RuntimeError(f'Unsupported GPU capability for installed PyTorch: {gpu_names}')
device = 'cuda'
torch.cuda.reset_peak_memory_stats()
wrapper = BifoldS2Inference.from_pretrained(BigEarthNetv2_0_ImageClassifier, cache_dir=Path('/kaggle/working/hf-cache'), allow_network=True)
summary = freeze_s2_classifier_head(wrapper.model, official_class_order=tuple(NEW_LABELS))
if (summary.trainable_parameter_count, summary.frozen_parameter_count) != (38931, 23529984):
    raise RuntimeError(f'Pinned parameter counts changed: {summary}')
config = Phase4FTrainingConfig()
train_dataset = Phase4FS2Dataset(manifest_path=EXPERIMENT_DIR / 'split_manifest.json', dataset_root=DATASET_ROOT, split='train')
validation_dataset = Phase4FS2Dataset(manifest_path=EXPERIMENT_DIR / 'split_manifest.json', dataset_root=DATASET_ROOT, split='validation')
if (len(train_dataset), len(validation_dataset)) != (12000, 3000):
    raise RuntimeError('Frozen Phase 4F split counts changed')
generator = torch.Generator().manual_seed(config.random_seed)
loader_args = dict(batch_size=config.batch_size, num_workers=4, pin_memory=True, persistent_workers=True)
train_loader = DataLoader(train_dataset, shuffle=True, generator=generator, **loader_args)
validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_args)
fit = fit_head_only(wrapper.model, train_batches=train_loader, validation_batches=validation_loader, config=config, device=device)
output_dir = Path('/kaggle/working/satquery-output') / os.environ['SATQUERY_REMOTE_OUTPUT']
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / 'phase4f_s2_head.safetensors'
head_state = {name: tensor.detach().cpu().contiguous() for name, tensor in wrapper.model.model.vision_encoder.fc.state_dict().items()}
save_file(head_state, str(checkpoint_path))
checkpoint_sha = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
registration = wrapper.registration
evaluation_provenance = Phase4EProvenance(experiment_name=os.environ['SATQUERY_EXPERIMENT_NAME'], git_sha=os.environ['SATQUERY_GIT_REF'], modality='s2', model_id=registration.model_id, model_revision=registration.revision, checkpoint_sha256=registration.checkpoint_sha256, preprocessing_profile=registration.preprocessing_profile, frozen_manifest_sha256=readiness['frozen_manifest_sha256'], materialized_package_sha256=readiness['modalities']['s2']['package_sha256'], threshold=0.5)
evaluation_batches = iter_validation_batches(manifest_path=EXPERIMENT_DIR / 'split_manifest.json', dataset_root=DATASET_ROOT, profile=wrapper.profile, batch_size=config.batch_size, device=device)
metrics, predictions = evaluate_validation_batches(evaluation_batches, wrapper, provenance=evaluation_provenance, gate_paths=Phase4DGatePaths(readiness=READINESS, raster_audit=EXPERIMENT_DIR / 'results/representative_raster_audit.json', preprocessing_contract=EXPERIMENT_DIR / 'bifold_contract.json', manifest=EXPERIMENT_DIR / 'split_manifest.json'))
runtime_seconds = time.perf_counter() - phase4f_started
peak_bytes = max(torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count()))
provenance = Phase4FProvenance(git_sha=os.environ['SATQUERY_GIT_REF'], dirty_worktree=_RUNNER_META['dirty_worktree'], kaggle_experiment=os.environ['SATQUERY_EXPERIMENT_NAME'], kaggle_kernel='satquery-phase4f-bifold-s2-head-adaptation', runtime_seconds=runtime_seconds, device=', '.join(gpu_names), cuda_version=torch.version.cuda, torch_version=torch.__version__, python_version=platform.python_version(), peak_gpu_memory_bytes=peak_bytes, model_id=registration.model_id, model_revision=registration.revision, base_checkpoint_sha256=registration.checkpoint_sha256, materialized_package_sha256=readiness['modalities']['s2']['package_sha256'], checkpoint_output_sha256=checkpoint_sha, manifest_sha256=readiness['frozen_manifest_sha256'], preprocessing_profile=registration.preprocessing_profile, test_accessed=False)
map_delta = metrics.macro_average_precision - float(FROZEN_PHASE4E_S2_BASELINE['macro_average_precision'])
macro_f1_delta = metrics.macro_f1 - float(FROZEN_PHASE4E_S2_BASELINE['macro_f1'])
decision = 'ADAPTATION_USEFUL' if map_delta > 0 and macro_f1_delta >= -0.02 else 'ADAPTATION_NOT_USEFUL'
prediction_path = output_dir / 'phase4f_s2_validation_predictions.jsonl'
prediction_path.write_text(''.join(row.model_dump_json() + '\n' for row in predictions), encoding='utf-8')
training_result = {'schema_version': 1, 'status': 'TRAINING_COMPLETE', 'strategy': 'official_19_class_head_only', 'training_config': config.model_dump(mode='json'), 'parameters': {'trainable_names': list(summary.trainable_parameter_names), 'trainable': summary.trainable_parameter_count, 'frozen': summary.frozen_parameter_count}, 'epochs_completed': fit.epochs_completed, 'best_epoch': fit.best_epoch, 'train_loss': list(fit.train_loss), 'validation_loss': list(fit.validation_loss), 'validation_macro_average_precision': list(fit.validation_macro_average_precision), 'decision': decision, 'test_accessed': False}
validation_result = {'schema_version': 1, 'status': 'VALIDATION_EVALUATED', 'provenance': provenance.model_dump(mode='json'), 'baseline': dict(FROZEN_PHASE4E_S2_BASELINE), 'metrics': metrics.model_dump(mode='json'), 'deltas': {'macro_average_precision': map_delta, 'micro_f1': metrics.micro_f1 - float(FROZEN_PHASE4E_S2_BASELINE['micro_f1']), 'macro_f1': macro_f1_delta}, 'decision': decision, 'prediction_count': len(predictions), 'prediction_artifact': prediction_path.name}
checkpoint_manifest = {'schema_version': 1, 'format': 'safetensors', 'artifact': checkpoint_path.name, 'sha256': checkpoint_sha, 'base_model_id': registration.model_id, 'base_model_revision': registration.revision, 'base_checkpoint_sha256': registration.checkpoint_sha256, 'state_dict_keys': sorted(head_state), 'class_order': list(evaluation_provenance.class_order), 'trainable_parameter_count': summary.trainable_parameter_count, 'test_accessed': False}
runner_meta = {**_RUNNER_META, 'kernel_slug': 'satquery-phase4f-bifold-s2-head-adaptation', 'runtime_seconds': runtime_seconds, 'gpu_devices': gpu_names, 'cuda_version': torch.version.cuda, 'torch_version': torch.__version__, 'python_version': platform.python_version(), 'peak_gpu_memory_bytes_per_device_max': peak_bytes, 'checkpoint_output_sha256': checkpoint_sha, 'test_accessed': False}
for name, payload in [('phase4f_s2_training_result.json', training_result), ('phase4f_s2_validation_result.json', validation_result), ('phase4f_s2_adapter_or_checkpoint_manifest.json', checkpoint_manifest), ('phase4f_runner_meta.json', runner_meta)]:
    (output_dir / name).write_text(json.dumps(payload, indent=2) + '\n', encoding='utf-8')
print({'decision': decision, 'validation_mAP': metrics.macro_average_precision, 'mAP_delta': map_delta, 'test_accessed': False})
